# gTAO-Net smoke — single subject, protocol-free

First **didactic** run of the proposed multimodal architecture over **one** DIPSER bundle.
The subject is **auto-selected**: a cheap label scan (zip central directory only, no image
decode) picks the bundle with the widest plurality-label spread, so the run actually has a
usable label range. Goal: read the bundle, build Tchebichef front-ends, train gTAO-Net
(per-stream towers → gated cross-modal fusion → global input→output skip → CORAL head) and
log metrics after every important step in `metrics_gtao_smoke.json`.

This is **not** the benchmark protocol (no cross-validation, one subject): it exists to make
the architecture readable and to catch pipeline bugs before the real subject-disjoint grid.

Streams read from the bundle (when present, huge-or-None so we never fabricate data):
  face (2-D Tchebichef per frame + 1-D across time), facemesh, head pose,
  wrist watch channels (1-D across time).
Labels: 5 raters, change-points with nulls filtered before forward-fill (`filter_nulls`
convention), 1–5 collapsed to 3 ordinal levels, plurality vote at the window midpoint.

In [ ]:
import json
import math
import time
import zipfile
from pathlib import Path

import mpmath
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import cohen_kappa_score

# ---------------------------------------------------------------- config ----
INPUT_DIR = Path("/kaggle/input/dipseer-paper1-slice")
BUNDLE = None  # chosen by the label-spread scan; set SMOKE_SUBJECT to override
SMOKE_SUBJECT = "auto"   # or an explicit "group_01__experiment_01__subject_01.zipbin"
OUT = Path("/kaggle/working")
LOG_FILE = OUT / "metrics_gtao_smoke.json"
MODEL_FILE = OUT / "gtao_smoke.pt"

WINDOW_S = 4.0           # annotation protocol window (seconds)
FACE_SIZE = 64           # canonical face-crop side
FACE_K = 16              # 2-D Tchebichef order -> 16x16 coefficient map
SEQ_K = 6                # 1-D Tchebichef order across time
N_CLASSES = 3            # low / medium / high attention
EMB = 64                 # stream embedding size
HID = 128                # tower hidden layer
EPOCHS = 80
BATCH = 32
LR = 3e-3
SEED = 0
TRAIN_FRAC = 0.70        # temporal split inside the subject
N_PHYSIO_CHANNELS = 16   # cap on watch channels kept per window

torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


def log_step(step: str, **payload):
    """Append one traceability entry to the JSON document on disk."""
    metrics = json.loads(LOG_FILE.read_text()) if LOG_FILE.exists() else {}
    metrics["status"] = "running"
    metrics.setdefault("steps", {})[step] = {"ts": time.strftime("%Y-%m-%d %H:%M:%S"), **payload}
    LOG_FILE.write_text(json.dumps(metrics, indent=2))
    print("[log]", step, {k: v for k, v in payload.items() if not isinstance(v, (np.ndarray, list))})

## 1) Tchebichef basis and transforms

Recurrence in polynomial degree `n`, computed in arbitrary precision (mpmath) and only
cast to float32 *after* division by the norm — the raw coefficients overflow float64
past order ~15. Basis depends only on `(N, K)`, so it is cached.

In [ ]:
def _rho(n, N):
    r = mpmath.mpf(N)
    for k in range(1, n + 1):
        r *= N * N - k * k
    return r / (2 * n + 1)


def tchebichef_basis(N, K):
    with mpmath.workdps(50):
        t = [[mpmath.mpf(0)] * N for _ in range(K)]
        for x in range(N):
            t[0][x] = mpmath.mpf(1)
        if K >= 2:
            for x in range(N):
                t[1][x] = mpmath.mpf(2 * x - N + 1)
        for n in range(2, K):
            a = mpmath.mpf(2 * n - 1)
            c = mpmath.mpf((n - 1) * (N * N - (n - 1) * (n - 1)))
            for x in range(N):
                t[n][x] = (a * (2 * x - N + 1) * t[n - 1][x] - c * t[n - 2][x]) / n
        B = np.zeros((N, K), dtype=np.float32)
        for n in range(K):
            norm = mpmath.sqrt(_rho(n, N))
            for x in range(N):
                B[x, n] = float(t[n][x] / norm)
    return B


_BASIS = {}


def basis(N, K):
    key = (N, K)
    if key not in _BASIS:
        _BASIS[key] = tchebichef_basis(N, K)
    return _BASIS[key]


def transform_2d(img, K):
    """T = B^T img B — separable, two GEMMs."""
    B = basis(img.shape[-1], K)
    return (B.T @ img) @ B


def transform_1d(x, K):
    """1-D moments along the trailing axis."""
    B = basis(x.shape[-1], K)
    return x @ B

## 2) Bundle reader → labelled 4s windows

* The `.zipbin` is a plain zip; we extract it to the working dir so `/labels`, `/metadata`
  and `/watch_sensors` land on disk.
* Rater change-points: nulls dropped *before* forward-fill, values collapse 5→3, the
  window label is the plurality of the 5 raters at the window midpoint.
* A window with no votes yet is skipped; a stream missed for too few frames is `None`.
* A stream entirely absent from the bundle is simply not added to the model.

In [ ]:
def parse_ts(s):
    s = str(s).strip().replace(".", ":").replace("-", ":").replace("_", ":")
    p = [x for x in s.split(":") if x != ""]
    h, m = int(p[0]), int(p[1])
    sec = int(p[2]) if len(p) > 2 else 0
    us = float(p[3].ljust(6, "0")) / 1e6 if len(p) > 3 and p[3] else 0.0
    return h * 3600 + m * 60 + sec + us


HEAD_POSE_KEYS = ("pitch", "yaw", "roll")


def read_facemesh(meta):
    person = meta.get("person") or {}
    fm = (person.get("face") or {}).get("facemesh")
    if not isinstance(fm, list) or not fm:
        return None
    try:
        pts = np.array([[p["x"], p["y"], p["z"]] for p in fm], dtype=np.float32)
    except (KeyError, TypeError):
        return None
    return pts if pts.ndim == 2 and pts.shape[1] == 3 else None


def read_pose(meta):
    pose = (meta.get("person") or {}).get("pose")
    if not isinstance(pose, list) or not pose:
        return None
    try:
        pts = np.array([[p["x"], p["y"], p["z"]] for p in pose], dtype=np.float32)
    except (KeyError, TypeError):
        return None
    return pts if pts.ndim == 2 and pts.shape[1] == 3 else None


def read_head_pose(meta):
    hp = meta.get("head_pose") if isinstance(meta.get("head_pose"), dict) else None
    if hp is None:
        try:
            hp = meta["person"]["face"]["headpose"]["pose"]
        except (KeyError, TypeError):
            return None
    if not isinstance(hp, dict):
        return None
    try:
        return np.array([float(hp[k]) for k in HEAD_POSE_KEYS], dtype=np.float32)
    except (KeyError, TypeError, ValueError):
        return None


def norm_watch_channel(sensor, fields):
    s = sensor.lower()
    if "hr" in s or "heart" in s:
        return {"value0": "HeartRate.bpm"}
    if "accel" in s:
        axes = ("x", "y", "z", "w")
        return {f"value{i}": f"Accel.{axes[i]}" for i in range(len(fields)) if i < 4}
    if "gyr" in s:
        axes = ("x", "y", "z", "w")
        return {f"value{i}": f"Gyro.{axes[i]}" for i in range(len(fields)) if i < 4}
    if "rot" in s:
        axes = ("w", "x", "y", "z")
        return {f"value{i}": f"Rot.{axes[i]}" for i in range(len(fields)) if i < 4}
    if "light" in s:
        return {"value0": "Light.lux"}
    return {f: f"{sensor}.{f}" for f in fields}


def read_watch_sensors(watch_dir):
    """{channel: (times, values)} using the canonical channel names."""
    out = {}
    if not watch_dir.is_dir():
        return out
    for f in sorted(watch_dir.iterdir()):
        if "__" in f.name or f.suffix != ".json":
            continue
        try:
            blob = json.loads(f.read_text())
        except json.JSONDecodeError:
            continue
        data = blob.get("data") if isinstance(blob, dict) else None
        if not isinstance(data, dict):
            continue
        for sensor, records in data.items():
            if not isinstance(records, list):
                continue
            for s in records:
                if not isinstance(s, dict):
                    continue
                tkey = next((k for k in s if "time" in k.lower()), None)
                if tkey is None:
                    continue
                raw_t = s[tkey]
                t = raw_t if isinstance(raw_t, (int, float)) else parse_ts(raw_t)
                fields = sorted(k for k in s if k != tkey and isinstance(s[k], (int, float)))
                mapping = norm_watch_channel(str(sensor), fields)
                for vf in fields:
                    key = mapping.get(vf, f"{sensor}.{vf}")
                    ch = out.setdefault(key, ([], []))
                    ch[0].append(t)
                    ch[1].append(float(s[vf]))
    result = {}
    for key, (ts, vs) in out.items():
        order = np.argsort(np.asarray(ts, dtype=np.float64), kind="mergesort")
        result[key] = (np.asarray(ts, dtype=np.float64)[order],
                       np.asarray(vs, dtype=np.float32)[order])
    return result


def read_rater_labels(labels_dir):
    """{rater: [(ts, attention)]}, null change-points dropped (filter_nulls)."""
    out = {}
    if not labels_dir.is_dir():
        return out
    for f in sorted(labels_dir.iterdir()):
        if "__" in f.name or f.suffix != ".json":
            continue
        try:
            raw = json.loads(f.read_text())
        except json.JSONDecodeError:
            continue
        if not isinstance(raw, list):
            continue
        cps = []
        for e in raw:
            if not isinstance(e, dict):
                continue
            v = e.get("attention")
            if v is None:
                continue
            cps.append((parse_ts(e["datetime"]), int(v)))
        out[f.stem] = sorted(cps, key=lambda r: r[0])
    return out


def value_at(cps, t):
    """Most recent change-point value at or before t; None before the first."""
    best = None
    for ts, v in cps:
        if ts <= t:
            best = v
        else:
            break
    return best


def collapse_5_to_3(v):
    return 0 if v <= 2 else (1 if v == 3 else 2)


def majority_vote(votes, n_classes=N_CLASSES):
    vs = [v for v in votes if v is not None]
    if not vs:
        return None
    return int(np.bincount(np.asarray(vs, dtype=int), minlength=n_classes).argmax())


def unpack_bundle(path):
    root = OUT / "bundle_tmp"
    root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as zf:
        for n in zf.namelist():
            if "/" in n:
                (root / n).parent.mkdir(parents=True, exist_ok=True)
            (root / n).write_bytes(zf.read(n))
    return root


def build_frames(subject_root):
    frames = []
    for img in sorted((subject_root / "images").glob("*.png")):
        frames.append((parse_ts(img.stem), img))
    return sorted(frames, key=lambda r: r[0])


def build_windows(subject_root, frames):
    """Time-aligned 4s windows with a plurality label at the midpoint."""
    raters = read_rater_labels(subject_root / "labels")
    if not raters:
        return []
    frame_ts = np.array([t for t, _ in frames], dtype=np.float64)
    windows = []
    t0, t1 = float(frame_ts[0]), float(frame_ts[-1])
    t = t0
    while t + WINDOW_S <= t1 + 1e-9:
        mid = t + WINDOW_S / 2
        votes = [collapse_5_to_3(value_at(cps, mid)) for cps in raters.values()]
        label = majority_vote(votes)
        if label is None:
            t += WINDOW_S
            continue
        lo, hi = np.searchsorted(frame_ts, [t, t + WINDOW_S])
        idx = list(range(int(lo), int(hi)))
        if not idx:
            t += WINDOW_S
            continue
        windows.append({"t_start": t, "t_end": t + WINDOW_S,
                        "frames": [frames[i] for i in idx],
                        "frame_idx": idx, "label": label})
        t += WINDOW_S
    return windows

## 3) Per-window Tchebichef front-ends

* **face**: 2-D Tchebichef per frame (`16x16`) then 1-D Tchebichef across frames
  → `(FACE_K²·SEQ_K)`.
* **facemesh / pose / head-pose**: 1-D Tchebichef of each coordinate trajectory.
* **physio**: 1-D Tchebichef per watch channel (channels must have ≥1 sample in
  every window, so the dimension is fixed).
* Sequences shorter than `SEQ_K` are edge-padded (the basis needs `K ≤ T`).
* Policy: streams present in ≥95% of windows enter the model; windows missing a
  kept stream are **dropped**, never zero-filled — absence stays absence.

In [ ]:
def pad_time(arr, k):
    """Edge-pad the time axis so 1-D moments of order k always exist (T >= k)."""
    if arr.shape[0] >= k:
        return arr
    if arr.shape[0] == 0:
        raise ValueError("empty time axis")
    return np.concatenate([arr, np.repeat(arr[-1:], k - arr.shape[0], axis=0)], axis=0)


def encode_window(w, subject_root, physio, physio_channels):
    faces, fm_seq, pose_seq, hp_seq = [], [], [], []
    for ts, img_path in w["frames"]:
        stem = img_path.stem
        img = np.asarray(Image.open(img_path).convert("L").resize((FACE_SIZE, FACE_SIZE)),
                         dtype=np.float32) / 255.0
        faces.append(transform_2d(img, FACE_K))          # (16, 16) per frame
        meta_path = subject_root / "metadata" / f"{stem}.json"
        if not meta_path.exists():
            continue
        meta = json.loads(meta_path.read_text())
        fm = read_facemesh(meta)
        if fm is not None:
            fm_seq.append(fm)
        pose = read_pose(meta)
        if pose is not None:
            pose_seq.append(pose)
        hp = read_head_pose(meta)
        if hp is not None:
            hp_seq.append(hp)

    if not faces:
        return None
    # every window encodes to the same shape: sequences shorter than SEQ_K are edge-padded
    face_seq = pad_time(np.stack(faces), SEQ_K)                 # (T, 16, 16)
    feat = {"face": transform_1d(
        np.moveaxis(face_seq, 0, -1).reshape(-1, face_seq.shape[0]), SEQ_K).ravel()}
    if fm_seq:
        fm_t = pad_time(np.stack(fm_seq), SEQ_K)                # (T, 478, 3)
        feat["facemesh"] = transform_1d(
            np.moveaxis(fm_t, 0, -1).reshape(478 * 3, fm_t.shape[0]), SEQ_K).ravel()
    if pose_seq:
        pose_t = pad_time(np.stack(pose_seq), SEQ_K)
        feat["pose"] = transform_1d(
            np.moveaxis(pose_t, 0, -1).reshape(33 * 3, pose_t.shape[0]), SEQ_K).ravel()
    if hp_seq:
        hp_t = pad_time(np.stack(hp_seq), SEQ_K)                # (T, 3)
        feat["head_pose"] = transform_1d(
            np.moveaxis(hp_t, 0, -1).reshape(3, hp_t.shape[0]), SEQ_K).ravel()

    # physio: fixed channel list; slice to [t_start, t_end), 1-D Tchebichef per channel
    # (edge-pad the *sample* axis so the basis K <= N always holds)
    ch_feats = []
    for ch in physio_channels:
        pts, vals = physio[ch]
        sel = (pts >= w["t_start"]) & (pts < w["t_end"])
        x = vals[sel]
        if len(x) < SEQ_K:
            x = np.pad(x, (0, SEQ_K - len(x)), mode="edge")
        ch_feats.append(transform_1d(x[None, :], SEQ_K).ravel())
    if ch_feats:
        feat["physio"] = np.concatenate(ch_feats)
    return feat

In [ ]:
import os

print("=" * 70)
print("STEP: data -> locate bundle, scan labels, pick subject")


def locate_dataset_root():
    for cand in ("/kaggle/input/dipseer-paper1-slice",
                 "/kaggle/input/datasets/dipseer-paper1-slice",
                 "/kaggle/input/datasets/zakariamakhas/dipseer-paper1-slice"):
        if any(Path(cand).glob("*.zipbin")):
            return Path(cand)
    for dp, _, fns in os.walk("/kaggle/input"):
        if any(f.endswith(".zipbin") for f in fns):
            return Path(dp)
    return None


def scan_bundle_labels(path):
    """Plurality-label histogram read from the zip central directory (cheap)."""
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        ts = sorted(parse_ts(n.rsplit("/", 1)[-1][:-4])
                    for n in names if n.startswith("images/") and n.endswith(".png"))
        if not ts:
            return None
        raters = {}
        for n in names:
            if n.startswith("labels/") and n.endswith(".json"):
                try:
                    raw = json.loads(zf.read(n))
                except json.JSONDecodeError:
                    continue
                if not isinstance(raw, list):
                    continue
                cps = [(parse_ts(e["datetime"]), int(e["attention"]))
                       for e in raw if isinstance(e, dict) and e.get("attention") is not None]
                raters[n.rsplit("/", 1)[-1][:-5]] = sorted(cps, key=lambda r: r[0])
        hist = [0, 0, 0]
        t = ts[0]
        while t + WINDOW_S <= ts[-1] + 1e-9:
            votes = [collapse_5_to_3(value_at(cps, t + WINDOW_S / 2)) for cps in raters.values()]
            lab = majority_vote(votes)
            if lab is not None:
                hist[lab] += 1
            t += WINDOW_S
        return tuple(hist)


def spread_score(h):
    tot = float(sum(h))
    if tot == 0:
        return (0, 0, 0.0)
    nonzero = [c for c in h if c > 0]
    p = [c / tot for c in h]
    ent = -sum(pi * math.log(pi) for pi in p if pi > 0)
    return (len(nonzero) if len(nonzero) >= 2 else 0, min(nonzero), round(ent, 4))


t0 = time.time()
INPUT_DIR = locate_dataset_root()
while INPUT_DIR is None and time.time() - t0 < 600:
    print("waiting for dataset mount ...")
    time.sleep(5)
    INPUT_DIR = locate_dataset_root()
if INPUT_DIR is None:
    raise FileNotFoundError("no *.zipbin under /kaggle/input — is the dataset attached?")

if BUNDLE is None:
    scan = {}
    for b in sorted(INPUT_DIR.glob("*.zipbin")):
        h = scan_bundle_labels(b)
        if h is not None:
            scan[b.name] = list(h)
    if SMOKE_SUBJECT == "auto":
        BUNDLE = max(scan, key=lambda n: spread_score(scan[n]))
    else:
        BUNDLE = SMOKE_SUBJECT
    chosen = spread_score(scan[BUNDLE])
    log_step("subject_scan", label_histograms=scan, chosen=BUNDLE,
             chosen_spread_score=chosen, wall_s=round(time.time() - t0, 2))
    print("label scan (bundle -> [low, mid, high]):")
    for n, h in scan.items():
        print("  ", n, h, "score", spread_score(h))

bundle_path = INPUT_DIR / BUNDLE
assert bundle_path.exists(), f"{bundle_path} missing"
print(f"chosen bundle: {BUNDLE}")

t_start = time.time()
subject_root = unpack_bundle(str(bundle_path))
frames = build_frames(subject_root)
windows = build_windows(subject_root, frames)
physio = read_watch_sensors(subject_root / "watch_sensors")
log_step("data_loaded",
         bundle=BUNDLE, n_frames=len(frames), n_windows=len(windows),
         physio_channels=sorted(physio.keys()),
         wall_s=round(time.time() - t_start, 2))
print("frames:", len(frames), "windows:", len(windows), "physio channels:", sorted(physio))

In [ ]:
print("=" * 70)
print("STEP: features -> encode windows")
t_start = time.time()

# keep only watch channels with >=1 sample in EVERY window -> fixed physio dimension
physio_channels = [ch for ch in sorted(physio)[:N_PHYSIO_CHANNELS]
                   if all(np.any((physio[ch][0] >= w["t_start"]) & (physio[ch][0] < w["t_end"]))
                          for w in windows)]

pairs = []
for w in windows:
    f = encode_window(w, subject_root, physio, physio_channels)
    if f is not None:
        pairs.append((f, w["label"]))

# stream policy: keep streams present in >=95% of windows, then keep only the
# windows that carry ALL kept streams (never zero-fill a missing modality)
keys = sorted(set().union(*(f.keys() for f, _ in pairs)))
coverage = {k: round(sum(1 for f, _ in pairs if k in f) / len(pairs), 3) for k in keys}
keep_streams = [k for k, c in coverage.items() if c >= 0.95]
n_before = len(pairs)
pairs = [(f, y) for f, y in pairs if all(k in f for k in keep_streams)]
feats = [f for f, _ in pairs]
labels = [y for _, y in pairs]

log_step("features_built",
         n_windows_kept=len(feats), n_windows_dropped=n_before - len(feats),
         label_counts=[int(c) for c in np.bincount(labels, minlength=N_CLASSES)],
         stream_coverage_pct=coverage, kept_streams=keep_streams,
         physio_channels=physio_channels, wall_s=round(time.time() - t_start, 2))
print("kept windows:", len(feats), "streams:", keep_streams, "coverage:", coverage)
assert len(feats) > 0, "no windows survived the coverage filter"

## 4) gTAO-Net model

* Per-stream **towers** embed the raw Tchebichef vectors → `z_m`.
* **Gated cross-modal fusion**: self-attention over `{z_m}` (a stream delegates to the
  modalities it trusts), then a learned gate `α` pools the attended embeddings — a noisy
  stream can be driven toward `α ≈ 0`.
* **Global input→output skip**: per-stream *linear* maps from the raw coefficient vectors
  straight to the pre-head vector, added to the trunk output. Unattenuated residual path
  head → earliest layers: the vanishing-gradient fix (CORE-Net G-H).
* **CORAL head**: rank-consistent ordinal logits over `K−1` thresholds, trained with
  per-threshold BCE; prediction via cumulative-probability arithmetic.

In [ ]:
class StreamTower(nn.Module):
    def __init__(self, in_dim, hidden=HID, emb=EMB):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.GELU(),
                                 nn.Linear(hidden, emb), nn.LayerNorm(emb))

    def forward(self, x):          # (B, in_dim) -> (B, emb)
        return self.net(x)


class GatedCrossModalFusion(nn.Module):
    def __init__(self, n_streams, emb=EMB):
        super().__init__()
        self.emb = emb
        self.proj_q = nn.Linear(emb, emb, bias=False)
        self.proj_k = nn.Linear(emb, emb, bias=False)
        self.gate_net = nn.Sequential(nn.Linear(n_streams * emb, n_streams * emb),
                                      nn.GELU(), nn.Linear(n_streams * emb, n_streams))

    def forward(self, z):          # z: (B, n, emb) -> (fused, alpha)
        # cross-modal self-attention over the n modality embeddings
        q = self.proj_q(z)
        k = self.proj_k(z)
        attn = torch.softmax(q @ k.transpose(1, 2) / math.sqrt(self.emb), dim=-1)
        z2 = attn @ z
        # learned gate pools the attended embeddings (noisy stream -> alpha ~ 0)
        alpha = torch.softmax(self.gate_net(z2.reshape(z2.shape[0], -1)), dim=-1)
        fused = (alpha.unsqueeze(-1) * z2).sum(dim=1)
        return fused, alpha


class GlobalSkip(nn.Module):
    """Per-stream linear map from raw coefficients to the pre-head vector.

    One clean linear path per stream — no activations, so the gradient is never
    squashed between the head and the towers' first layers.
    """

    def __init__(self, stream_dims, emb=EMB):
        super().__init__()
        self.proj = nn.ModuleDict({k: nn.Linear(d, emb, bias=False)
                                   for k, d in stream_dims.items()})

    def forward(self, streams):
        return torch.stack([self.proj[k](x) for k, x in streams.items()]).sum(dim=0)


class CORALHead(nn.Module):
    def __init__(self, emb=EMB, num_classes=N_CLASSES):
        super().__init__()
        self.d = num_classes - 1
        self.w = nn.Linear(emb, 1)
        self.b = nn.Parameter(torch.linspace(0.5, -0.5, self.d))

    def forward(self, x):          # (B, emb) -> (B, d) threshold logits
        return self.w(x) + self.b

    def loss(self, logits, y):
        # rank-consistent by construction: one shared w, ordered thresholds b
        k = torch.arange(self.d, device=logits.device)[None, :]
        targets = (y[:, None] > k).float()
        return F.binary_cross_entropy_with_logits(logits, targets)

    def predict(self, logits):
        # cumulative survival -> per-class probability -> argmax
        pgt = torch.sigmoid(logits)
        ext = torch.cat([torch.ones_like(pgt[:, :1]), pgt,
                         torch.zeros_like(pgt[:, :1])], dim=1)
        p = (ext[:, :-1] - ext[:, 1:]).clamp(min=0)
        return (p / p.sum(dim=1, keepdim=True)).argmax(dim=1)


class GTAONet(nn.Module):
    def __init__(self, stream_dims, use_skip=True, emb=EMB):
        super().__init__()
        self.streams = sorted(stream_dims)
        self.towers = nn.ModuleDict({k: StreamTower(stream_dims[k]) for k in self.streams})
        self.fusion = GatedCrossModalFusion(len(self.streams))
        self.skip = GlobalSkip(stream_dims) if use_skip else None
        self.head = CORALHead()
        self.use_skip = use_skip

    def forward(self, streams):
        z = torch.stack([self.towers[k](streams[k]) for k in self.streams], dim=1)
        fused, alpha = self.fusion(z)
        if self.use_skip and self.skip is not None:
            fused = fused + self.skip(streams)
        return self.head(fused), alpha

In [ ]:
print("=" * 70)
print("STEP: model -> build")
stream_dims = {k: len(feats[0][k]) for k in sorted(feats[0])}
X = {k: np.stack([f[k] for f in feats]).astype(np.float32) for k in stream_dims}
Y = np.asarray(labels, dtype=np.int64)
n = len(Y)

model = GTAONet(stream_dims).to(device)
n_params = sum(p.numel() for p in model.parameters())
log_step("model_built", stream_dims=stream_dims, n_params=n_params, device=device)
print("stream_dims:", stream_dims)

## 5) Gradient probe (at initialization) + training

First 70% of windows in time → train, next 30% → validation.

The vanishing-gradient **probe** runs *before* any training, on one identical
batch, so both variants see comparable loss scale: gradient norms at the towers'
first layers with the global skip ON vs OFF, plus the gradient norm through the
skip's own projections (the clean input→head path, H3'). Measuring after training
would be confounded: once the trunk alone explains the output, toggling the skip
off changes the forward pass and inflates the OFF-side gradient.

In [ ]:
def to_torch(streams_dict):
    return {k: torch.as_tensor(v, device=device) for k, v in streams_dict.items()}


@torch.no_grad()
def evaluate(model, Xv, yv):
    logits, _ = model(Xv)
    pred = model.head.predict(logits).cpu().numpy()
    y = yv.cpu().numpy()
    try:
        qwk = float(cohen_kappa_score(y, pred, weights="quadratic"))
    except ValueError:
        qwk = 0.0
    if not np.isfinite(qwk):
        qwk = 0.0
    return {"qwk": round(qwk, 4),
            "acc": round(float((pred == y).mean()), 4),
            "mae": round(float(np.abs(pred - y).mean()), 4)}


def tower_first_grad_norm(model, streams, y, use_skip):
    """||dL/d(first Linear weight)|| across every tower, for the requested path."""
    model.use_skip = use_skip
    model.zero_grad()
    logits, _ = model(streams)
    loss = model.head.loss(logits, y)
    loss.backward()
    total = 0.0
    for tower in model.towers.values():
        w = tower.net[0].weight
        total += (w.grad ** 2).sum().item()
    return math.sqrt(total)


def skip_grad_norm(model, streams, y):
    """||dL/d(skip projection)|| — gradient through the single clean linear path."""
    model.use_skip = True
    model.zero_grad()
    logits, _ = model(streams)
    loss = model.head.loss(logits, y)
    loss.backward()
    total = 0.0
    for p in model.skip.proj.values():
        total += (p.weight.grad ** 2).sum().item()
    return math.sqrt(total)

In [ ]:
print("=" * 70)
print("STEP: train")
order = np.argsort([w["t_start"] for w in windows])
cut = max(1, int(n * TRAIN_FRAC))
train_idx, val_idx = order[:cut], order[cut:]

Xtr = to_torch({k: X[k][train_idx] for k in stream_dims})
Ytr = torch.as_tensor(Y[train_idx], device=device)
Xva = to_torch({k: X[k][val_idx] for k in stream_dims})
Yva = torch.as_tensor(Y[val_idx], device=device)

opt = torch.optim.Adam(model.parameters(), lr=LR)
log_step("split", n_train=len(train_idx), n_val=len(val_idx),
         train_balance=[int(c) for c in np.bincount(Y[train_idx], minlength=N_CLASSES)],
         val_balance=[int(c) for c in np.bincount(Y[val_idx], minlength=N_CLASSES)])

# --- vanishing-gradient probe, untrained model, identical batch -------------
print("STEP: probe -> gradient paths at init (skip ON vs OFF)")
k_probe = min(64, len(train_idx))
probe_x = to_torch({k: X[k][train_idx[:k_probe]] for k in stream_dims})
probe_y = torch.as_tensor(Y[train_idx[:k_probe]], device=device)
g_tower_on = tower_first_grad_norm(model, probe_x, probe_y, use_skip=True)
g_tower_off = tower_first_grad_norm(model, probe_x, probe_y, use_skip=False)
g_skip = skip_grad_norm(model, probe_x, probe_y)
ratio_skip_over_tower = g_skip / max(g_tower_off, 1e-9)
model.use_skip = True
log_step("grad_probe_init",
         tower_first_grad_skipON=g_tower_on, tower_first_grad_skipOFF=g_tower_off,
         skip_proj_grad=g_skip,
         ratio_skip_over_tower_OFF=round(ratio_skip_over_tower, 3))
print(f"grad norms  tower-ON = {g_tower_on:.4e}  tower-OFF = {g_tower_off:.4e}  "
      f"skip = {g_skip:.4e}  ratio(skip/tower-OFF) = {ratio_skip_over_tower:.3f}")

print(f"{'epoch':>5} {'train-loss':>10} {'train-QWK':>9} {'val-QWK':>8}   gate mean per stream")

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(train_idx))
    losses = []
    for s in range(0, len(train_idx), BATCH):
        idx = perm[s:s + BATCH]
        bx = {k: Xtr[k][idx] for k in stream_dims}
        by = Ytr[idx]
        opt.zero_grad()
        logits, _ = model(bx)
        loss = model.head.loss(logits, by)
        loss.backward()
        opt.step()
        losses.append(loss.item())

    if epoch % 10 == 0 or epoch == EPOCHS:
        model.eval()
        tr = evaluate(model, Xtr, Ytr)
        va = evaluate(model, Xva, Yva)
        with torch.no_grad():
            _, alpha = model(Xtr)
            gate = {s: round(float(alpha.mean(dim=0).cpu().numpy()[i]), 3)
                    for i, s in enumerate(model.streams)}
        log_step(f"epoch_{epoch:03d}", train_loss=round(float(np.mean(losses)), 4),
                 **{"train_" + k: v for k, v in tr.items()},
                 **{"val_" + k: v for k, v in va.items()}, gate_mean=gate)
        print(f"{epoch:5d} {np.mean(losses):10.4f} {tr['qwk']:9.3f} {va['qwk']:8.3f}   {gate}")

# final evaluation + robustness hint: zero the pose stream (a missing modality)
model.eval()
final = evaluate(model, Xva, Yva)
if "pose" in stream_dims:
    Xva_no_pose = {k: (torch.zeros_like(v) if k == "pose" else v)
                   for k, v in Xva.items()}
    final["val_qwk_pose_zeroed"] = evaluate(model, Xva_no_pose, Yva)["qwk"]

torch.save(model.state_dict(), MODEL_FILE)
final_clean = {k: v for k, v in final.items() if not isinstance(v, dict)}
log_step("final", **final_clean, gtao_run="subject_01_smoke", model_saved=str(MODEL_FILE))
print("final val:", final)

## 6) Summary

The metric document lives at `/kaggle/working/metrics_gtao_smoke.json` (one entry per step:
`data_loaded`, `features_built`, `model_built`, `split`, `grad_probe_init`, `epoch_*`, `final`),
weights at `/kaggle/working/gtao_smoke.pt`. **Protocol caveat**: a single subject, temporal
hold-out, no seeds — numbers here are *pipeline smoke*, not evidence. The subject-disjoint
cross-validation grid is the actual experiment.

In [ ]:
print("=" * 70)
print("STEP: summary")
metrics = json.loads(LOG_FILE.read_text())
metrics["status"] = "done"
metrics["finished_utc"] = time.strftime("%Y-%m-%d %H:%M:%S")
LOG_FILE.write_text(json.dumps(metrics, indent=2))
print("log steps:", list(metrics["steps"]))
print("final:", metrics["steps"].get("final"))